# Construindo a Arquitetura da CNN 

![Extracao de características](extracao_caracteristicas.png)

## 1. Importando as bibliotecas

In [139]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Convolution2D
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import Dense
from keras.layers import Dropout
from keras import utils
import numpy as np

## 2. Aquisição dos dados

In [140]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

In [141]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


## 3. Pré-processamento

In [142]:
X_train = X_train / 255.
X_test = X_test / 255.
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)
y_train = utils.to_categorical(y_train) # converte os rótulos para one-hot
y_test = utils.to_categorical(y_test) # mantém o mesmo formato para teste

In [143]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(60000, 28, 28, 1)
(60000, 10)
(10000, 28, 28, 1)
(10000, 10)


## 4. Arquitetura da CNN

![Arquitetura CNN](cnn_arquitetura_basica.png)

In [144]:
# Inicializando a CNN
classifier = Sequential()

#Camada de convolução
classifier.add(Convolution2D(32, kernel_size=(3,3), input_shape = (28, 28,1), activation = 'relu', padding='same', name = 'conv_1'))

#Camada de pooling
classifier.add(MaxPooling2D(pool_size=(2,2), strides=(2, 2), padding='same', name = 'pool_1'))

#Segunda camada convolucional
classifier.add(Convolution2D(64, kernel_size=(3,3), activation = 'relu', padding='same', name = 'conv_2'))


#Segunda camada de pooling
classifier.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same', name = 'pool_2'))


#Vetorizando os mapas de características do último pooling (camada de entrada)
classifier.add(Flatten())

#Dropout
classifier.add(Dropout(0.5))

#Camada totalmente conectada ou oculta
classifier.add(Dense(activation='relu', units=128, name = 'dense_1'))


#Camada de saída
classifier.add(Dense(activation='softmax', units=10,  name = 'classification'))

In [145]:
classifier.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1 (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling2D)           │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling2D)           │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_10 (Flatten)            │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification (Dense)          │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Treinando o modelo

In [146]:
#Parâmetros de treinamento
epochs = 50
batch_size = 50
validation_split=0.1

In [147]:
print(int(X_train.shape[0] * (1 - validation_split) / batch_size))

1080


In [148]:
classifier.compile(optimizer = 'adam', loss= 'categorical_crossentropy', metrics=['accuracy'])

checkpoint = keras.callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', verbose=1, save_best_only=True, mode='max', save_freq='epoch') 
earlystop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [149]:
history = classifier.fit(X_train, y_train, validation_split=validation_split, batch_size=batch_size, epochs=epochs, callbacks=[checkpoint,earlystop], verbose=1)

# Verificações do histórico de treino
print('épocas executadas:', len(history.history.get('loss', [])))
print('últimos val_loss:', history.history.get('val_loss', [])[-10:])
print('últimos val_accuracy:', history.history.get('val_accuracy', [])[-10:])
print('earlystop stopped_epoch:', getattr(earlystop, 'stopped_epoch', None))
print('earlystop best:', getattr(earlystop, 'best', None))

# Cálculo da melhor época (por val_accuracy e val_loss)
val_loss = history.history.get('val_loss', [])
val_acc = history.history.get('val_accuracy', [])
if len(val_acc) > 0:
    best_acc_idx = int(np.argmax(val_acc))
    print('melhor época (por val_accuracy):', best_acc_idx + 1, 'val_accuracy =', val_acc[best_acc_idx])
if len(val_loss) > 0:
    best_loss_idx = int(np.argmin(val_loss))
    print('melhor época (por val_loss):', best_loss_idx + 1, 'val_loss =', val_loss[best_loss_idx])

Epoch 1/50
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7576 - loss: 0.6558
Epoch 1: val_accuracy improved from None to 0.88383, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - accuracy: 0.8274 - loss: 0.4725 - val_accuracy: 0.8838 - val_loss: 0.3197
Epoch 2/50
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8775 - loss: 0.3308
Epoch 2: val_accuracy improved from 0.88383 to 0.89950, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.8818 - loss: 0.3216 - val_accuracy: 0.8995 - val_loss: 0.2737
Epoch 3/50
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8963 - loss: 0.2815
Epoch 3: val_accuracy improved from 0.89950 to 0.90600, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.8990 - loss:

## 6. Avaliando o modelo

In [150]:
best_model = keras.models.load_model("best_model.keras")

In [151]:
score = best_model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.2149616926908493
Test accuracy: 0.9258000254631042


𝐴𝑡𝑖𝑣𝑖𝑑𝑎𝑑𝑒:  Treinar e Avaliar a arquitetura adaptada para outro conjunto de dados.